In [1]:
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error

In [2]:
df_jan = pd.read_parquet('./data/yellow_tripdata_2022-01.parquet')
df_feb = pd.read_parquet('./data/yellow_tripdata_2022-02.parquet')

### Q1. Downloading the data

We'll use the same NYC taxi dataset, but instead of "Green Taxi Trip Records", we'll use "Yellow Taxi Trip Records".

Download the data for January and February 2022.

Read the data for January. How many columns are there?

In [3]:
df_jan.shape

(2463931, 19)

January data has 19 columns

### Q2. Computing duration

Now let's compute the duration variable. It should contain the duration of a ride in minutes.

What's the standard deviation of the trips duration in January?

In [4]:
df_jan['duration'] = df_jan.tpep_dropoff_datetime - df_jan.tpep_pickup_datetime
df_jan['duration'] = df_jan.duration.dt.total_seconds() / 60

In [5]:
df_jan['duration'].std()

46.44530513776802

the standard deviation of the trips duration in January is 46.45

### Q3. Dropping outliers

Next, we need to check the distribution of the duration variable. There are some outliers. Let's remove them and keep only the records where the duration was between 1 and 60 minutes (inclusive).

What fraction of the records left after you dropped the outliers?

In [6]:
df_jan_clean = df_jan[(df_jan.duration >= 1) & (df_jan.duration <= 60)].copy()

In [7]:
df_jan_clean.shape[0]/df_jan.shape[0]

0.9827547930522406

The fraction of the records left after dropping outliers is 98%

### Q4. One-hot encoding

Let's apply one-hot encoding to the pickup and dropoff location IDs. We'll use only these two features for our model.

    - Turn the dataframe into a list of dictionaries
    - Fit a dictionary vectorizer
    - Get a feature matrix from it

What's the dimensionality of this matrix (number of columns)?

In [8]:
categorical = ['PULocationID', 'DOLocationID']

df_jan_clean[categorical] = df_jan_clean[categorical].fillna(-1).astype('int')

In [9]:
df_jan_clean[categorical] = df_jan_clean[categorical].astype('str')

In [10]:
train_dicts = df_jan_clean[categorical].to_dict(orient='records')

In [11]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts) 

In [12]:
X_train.shape

(2421440, 515)

In [13]:
y_train = df_jan_clean.duration.values

In [14]:
len(dv.feature_names_)

515

The dimensionality of this matrix is 515

### Q5. Training a model

Now let's use the feature matrix from the previous step to train a model.

    - Train a plain linear regression model with default parameters
    - Calculate the RMSE of the model on the training data

What's the RMSE on train?

In [15]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [16]:
y_pred = lr.predict(X_train)

In [17]:
mean_squared_error(y_train, y_pred, squared=False)

6.986190687384885

the RMSE on train set is 6.99

### Q6. Evaluating the model

Now let's apply this model to the validation dataset (February 2022).

What's the RMSE on validation?

In [19]:
df_feb['duration'] = df_feb.tpep_dropoff_datetime - df_feb.tpep_pickup_datetime
df_feb['duration'] = df_feb.duration.dt.total_seconds() / 60

df_feb_clean = df_feb[(df_feb.duration >= 1) & (df_feb.duration <= 60)].copy()

df_feb_clean[categorical] = df_feb_clean[categorical].fillna(-1).astype('int').astype('str')

In [20]:
val_dicts = df_feb_clean[categorical].to_dict(orient='records')

In [21]:
X_val = dv.transform(val_dicts) 

In [22]:
y_pred = lr.predict(X_val)

In [23]:
y_val = df_feb_clean.duration.values

In [24]:
mean_squared_error(y_val, y_pred, squared=False)

7.786408043158955

the RMSE on validation set is 7.79